In [8]:
print(selected_features)

['arpu_8', 'offnet_mou_8', 'loc_og_t2m_mou_7', 'loc_og_t2m_mou_8', 'loc_og_mou_7', 'loc_og_mou_8', 'total_og_mou_8', 'loc_ic_t2m_mou_7', 'loc_ic_t2m_mou_8', 'loc_ic_mou_7', 'loc_ic_mou_8', 'total_ic_mou_7', 'total_ic_mou_8', 'total_rech_num_8', 'total_rech_amt_8', 'max_rech_amt_8', 'last_day_rch_amt_8', 'aon', 'tenure_years', 'tenure_months', 'min_arpu', 'recent_arpu', 'arpu_change', 'avg_incoming_usage', 'recent_incoming_usage', 'incoming_usage_change', 'recent_outgoing_usage', 'outgoing_usage_change', 'recent_offnet_usage', 'activity_decline']


In [17]:
# ============================================================
# TELECOM CUSTOMER CHURN
# COMPLETE FEATURE ENGINEERING + MODEL TRAINING PIPELINE
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 1. LOAD RAW DATA
# ============================================================

DATA_PATH = (
    "C:/Users/Administrator/AI-ENINEERING/"
    "project1/data/raw/telecom_churn_data.csv"
)

df = pd.read_csv(DATA_PATH)

print("=" * 70)
print("RAW DATA")
print("=" * 70)

print("Shape:", df.shape)


# ============================================================
# 2. CREATE CHURN TARGET
# ============================================================

df["churn"] = (
    (df["total_ic_mou_9"].fillna(0) == 0) &
    (df["total_og_mou_9"].fillna(0) == 0) &
    (df["vol_2g_mb_9"].fillna(0) == 0) &
    (df["vol_3g_mb_9"].fillna(0) == 0)
).astype(int)


# ============================================================
# 3. REMOVE IDENTIFIERS
# ============================================================

identifier_columns = [
    "mobile_number",
    "circle_id"
]

df.drop(
    columns=[
        col
        for col in identifier_columns
        if col in df.columns
    ],
    inplace=True,
    errors="ignore"
)


# ============================================================
# 4. TENURE FEATURES
# ============================================================

if "aon" in df.columns:

    df["tenure_years"] = df["aon"] / 365

    df["tenure_months"] = df["aon"] / 30.44

    df["is_new_customer"] = (
        df["tenure_months"] < 6
    ).astype(int)


# ============================================================
# 5. ARPU FEATURES
# ============================================================

arpu_cols = [
    col
    for col in [
        "arpu_6",
        "arpu_7",
        "arpu_8"
    ]
    if col in df.columns
]

if arpu_cols:

    df["avg_arpu"] = df[arpu_cols].mean(axis=1)

    df["max_arpu"] = df[arpu_cols].max(axis=1)

    df["min_arpu"] = df[arpu_cols].min(axis=1)

    df["recent_arpu"] = df["arpu_8"]

    df["arpu_change"] = (
        df["arpu_8"] -
        df["arpu_6"]
    )

    df["arpu_change_pct"] = (
        (df["arpu_8"] - df["arpu_6"]) /
        (df["arpu_6"].abs() + 1e-6)
    )


# ============================================================
# 6. RECHARGE FEATURES
# ============================================================

recharge_amount_cols = [
    col
    for col in [
        "rech_amt_6",
        "rech_amt_7",
        "rech_amt_8"
    ]
    if col in df.columns
]

recharge_count_cols = [
    col
    for col in [
        "rech_num_6",
        "rech_num_7",
        "rech_num_8"
    ]
    if col in df.columns
]

if recharge_amount_cols:

    df["avg_recharge_amount"] = (
        df[recharge_amount_cols].mean(axis=1)
    )

    df["total_recharge_amount"] = (
        df[recharge_amount_cols].sum(axis=1)
    )

    df["recent_recharge_amount"] = (
        df["rech_amt_8"]
    )

    df["recharge_change"] = (
        df["rech_amt_8"] -
        df["rech_amt_6"]
    )


if recharge_count_cols:

    df["avg_recharge_frequency"] = (
        df[recharge_count_cols].mean(axis=1)
    )

    df["total_recharge_count"] = (
        df[recharge_count_cols].sum(axis=1)
    )


# ============================================================
# 7. VOICE USAGE FEATURES
# ============================================================

incoming_cols = [
    col
    for col in [
        "total_ic_mou_6",
        "total_ic_mou_7",
        "total_ic_mou_8"
    ]
    if col in df.columns
]

outgoing_cols = [
    col
    for col in [
        "total_og_mou_6",
        "total_og_mou_7",
        "total_og_mou_8"
    ]
    if col in df.columns
]

if incoming_cols:

    df["avg_incoming_usage"] = (
        df[incoming_cols].mean(axis=1)
    )

    df["recent_incoming_usage"] = (
        df["total_ic_mou_8"]
    )

    df["incoming_usage_change"] = (
        df["total_ic_mou_8"] -
        df["total_ic_mou_6"]
    )


if outgoing_cols:

    df["avg_outgoing_usage"] = (
        df[outgoing_cols].mean(axis=1)
    )

    df["recent_outgoing_usage"] = (
        df["total_og_mou_8"]
    )

    df["outgoing_usage_change"] = (
        df["total_og_mou_8"] -
        df["total_og_mou_6"]
    )


# ============================================================
# 8. DATA USAGE FEATURES
# ============================================================

two_g_cols = [
    col
    for col in [
        "vol_2g_mb_6",
        "vol_2g_mb_7",
        "vol_2g_mb_8"
    ]
    if col in df.columns
]

three_g_cols = [
    col
    for col in [
        "vol_3g_mb_6",
        "vol_3g_mb_7",
        "vol_3g_mb_8"
    ]
    if col in df.columns
]

if two_g_cols:

    df["avg_2g_usage"] = (
        df[two_g_cols].mean(axis=1)
    )

    df["total_2g_usage"] = (
        df[two_g_cols].sum(axis=1)
    )

    df["recent_2g_usage"] = (
        df["vol_2g_mb_8"]
    )


if three_g_cols:

    df["avg_3g_usage"] = (
        df[three_g_cols].mean(axis=1)
    )

    df["total_3g_usage"] = (
        df[three_g_cols].sum(axis=1)
    )

    df["recent_3g_usage"] = (
        df["vol_3g_mb_8"]
    )


# ============================================================
# 9. TOTAL DATA USAGE
# ============================================================

if (
    "avg_2g_usage" in df.columns
    and
    "avg_3g_usage" in df.columns
):

    df["avg_total_data_usage"] = (
        df["avg_2g_usage"] +
        df["avg_3g_usage"]
    )


# ============================================================
# 10. ONNET / OFFNET FEATURES
# ============================================================

onnet_cols = [
    col
    for col in [
        "onnet_mou_6",
        "onnet_mou_7",
        "onnet_mou_8"
    ]
    if col in df.columns
]

offnet_cols = [
    col
    for col in [
        "offnet_mou_6",
        "offnet_mou_7",
        "offnet_mou_8"
    ]
    if col in df.columns
]

if onnet_cols:

    df["avg_onnet_usage"] = (
        df[onnet_cols].mean(axis=1)
    )

    df["recent_onnet_usage"] = (
        df["onnet_mou_8"]
    )


if offnet_cols:

    df["avg_offnet_usage"] = (
        df[offnet_cols].mean(axis=1)
    )

    df["recent_offnet_usage"] = (
        df[offnet_cols].mean(axis=1)
    )

    df["recent_offnet_usage"] = (
        df["offnet_mou_8"]
    )


# ============================================================
# 11. ACTIVITY DECLINE
# ============================================================

if (
    "total_ic_mou_6" in df.columns
    and
    "total_og_mou_6" in df.columns
    and
    "total_ic_mou_8" in df.columns
    and
    "total_og_mou_8" in df.columns
):

    df["activity_decline"] = (
        df["total_ic_mou_6"].fillna(0)
        +
        df["total_og_mou_6"].fillna(0)
        -
        df["total_ic_mou_8"].fillna(0)
        -
        df["total_og_mou_8"].fillna(0)
    )


# ============================================================
# 12. REMOVE MONTH 9 FEATURES
# ============================================================

month_9_columns = [
    col
    for col in df.columns
    if "_9" in col
]

df.drop(
    columns=month_9_columns,
    inplace=True,
    errors="ignore"
)


# ============================================================
# 13. SEPARATE X AND y
# ============================================================

y = df["churn"]

X = df.drop(
    columns=["churn"]
)


# ============================================================
# 14. KEEP NUMERIC FEATURES
# ============================================================

X = X.select_dtypes(
    include=np.number
)


# ============================================================
# 15. REMOVE FEATURES WITH >60% MISSING VALUES
# ============================================================

missing_ratio = X.isnull().mean()

high_missing_columns = (
    missing_ratio[
        missing_ratio > 0.60
    ].index
)

X.drop(
    columns=high_missing_columns,
    inplace=True
)


print("\nFeatures after missing-value filtering:")
print(X.shape[1])


# ============================================================
# 16. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ============================================================
# 17. IMPUTE DATA
# ============================================================
# IMPORTANT:
# SelectKBest cannot work with NaN values.
# Therefore we MUST impute before feature selection.
# ============================================================

imputer = SimpleImputer(
    strategy="median"
)

X_train_imputed = imputer.fit_transform(
    X_train
)

X_test_imputed = imputer.transform(
    X_test
)


print("\nMissing values handled successfully.")


# ============================================================
# 18. FEATURE SELECTION
# ============================================================

selector = SelectKBest(
    score_func=f_classif,
    k=30
)

X_train_selected = selector.fit_transform(
    X_train_imputed,
    y_train
)

X_test_selected = selector.transform(
    X_test_imputed
)


# ============================================================
# 19. GET SELECTED FEATURE NAMES
# ============================================================

selected_mask = selector.get_support()

selected_features = X_train.columns[
    selected_mask
].tolist()

print("\n" + "=" * 70)
print("SELECTED FEATURES")
print("=" * 70)

for feature in selected_features:
    print(" •", feature)

print("\nNumber of selected features:")
print(len(selected_features))


# ============================================================
# 20. CREATE SELECTED DATAFRAMES
# ============================================================

X_train_selected = pd.DataFrame(
    X_train_selected,
    columns=selected_features,
    index=X_train.index
)

X_test_selected = pd.DataFrame(
    X_test_selected,
    columns=selected_features,
    index=X_test.index
)


# ============================================================
# 21. CREATE FINAL DEPLOYMENT PIPELINE
# ============================================================
# This pipeline starts AFTER feature selection.
#
# API input:
# 30 selected features
#
# Then:
# 30 features
#       ↓
#    Imputer
#       ↓
#    Scaler
#       ↓
# Logistic Regression
# ============================================================

final_pipeline = Pipeline([
    
    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),
    
    (
        "scaler",
        RobustScaler()
    ),
    
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])


# ============================================================
# 22. TRAIN FINAL PIPELINE
# ============================================================

final_pipeline.fit(
    X_train_selected,
    y_train
)

print("\nFinal pipeline trained successfully.")


# ============================================================
# 23. MAKE PREDICTIONS
# ============================================================

predictions = final_pipeline.predict(
    X_test_selected
)

probabilities = final_pipeline.predict_proba(
    X_test_selected
)[:, 1]


# ============================================================
# 24. MODEL EVALUATION
# ============================================================

accuracy = accuracy_score(
    y_test,
    predictions
)

print("\n" + "=" * 70)
print("MODEL PERFORMANCE")
print("=" * 70)

print("\nAccuracy:")
print(round(accuracy, 4))

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        predictions
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        predictions
    )
)


# ============================================================
# 25. VERIFY PIPELINE INPUT SIZE
# ============================================================

print("\n" + "=" * 70)
print("PIPELINE VERIFICATION")
print("=" * 70)

print(
    "Pipeline expects:",
    final_pipeline.n_features_in_,
    "features"
)

print(
    "Selected features:",
    len(selected_features)
)


# ============================================================
# 26. SAVE FINAL PIPELINE
# ============================================================

MODEL_DIR = (
    "C:/Users/Administrator/AI-ENINEERING/"
    "project1/models"
)

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "churn_final_pipeline.joblib"
)

joblib.dump(
    final_pipeline,
    MODEL_PATH
)


# ============================================================
# 27. SAVE SELECTED FEATURE NAMES
# ============================================================

FEATURE_PATH = os.path.join(
    MODEL_DIR,
    "selected_features.joblib"
)

joblib.dump(
    selected_features,
    FEATURE_PATH
)


# ============================================================
# 28. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 70)
print("TRAINING COMPLETED SUCCESSFULLY")
print("=" * 70)

print("\nModel saved to:")
print(MODEL_PATH)

print("\nSelected features saved to:")
print(FEATURE_PATH)

print("\nFinal model expects:")
print(
    final_pipeline.n_features_in_,
    "features"
)

print("\n" + "=" * 70)

RAW DATA
Shape: (99999, 226)


C:\Users\Administrator\AppData\Local\Temp\ipykernel_18664\1116916993.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["churn"] = (
C:\Users\Administrator\AppData\Local\Temp\ipykernel_18664\1116916993.py:80: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["tenure_years"] = df["aon"] / 365
C:\Users\Administrator\AppData\Local\Temp\ipykernel_18664\1116916993.py:82: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joini


Features after missing-value filtering:
161

Missing values handled successfully.


c:\Users\Administrator\AI-ENINEERING\project1\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [ 0  1  2 42 43 44 81 82 83] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\Administrator\AI-ENINEERING\project1\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw



SELECTED FEATURES
 • arpu_8
 • offnet_mou_8
 • loc_og_t2m_mou_7
 • loc_og_t2m_mou_8
 • loc_og_mou_7
 • loc_og_mou_8
 • total_og_mou_8
 • loc_ic_t2m_mou_7
 • loc_ic_t2m_mou_8
 • loc_ic_mou_7
 • loc_ic_mou_8
 • total_ic_mou_7
 • total_ic_mou_8
 • total_rech_num_8
 • total_rech_amt_8
 • max_rech_amt_8
 • last_day_rch_amt_8
 • aon
 • tenure_years
 • tenure_months
 • min_arpu
 • recent_arpu
 • arpu_change
 • avg_incoming_usage
 • recent_incoming_usage
 • incoming_usage_change
 • recent_outgoing_usage
 • outgoing_usage_change
 • recent_offnet_usage
 • activity_decline

Number of selected features:
30

Final pipeline trained successfully.

MODEL PERFORMANCE

Accuracy:
0.7944

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.79      0.87     17962
           1       0.31      0.82      0.45      2038

    accuracy                           0.79     20000
   macro avg       0.64      0.81      0.66     20000
weighted avg       0.91   